# The neutrino-mass ladder

Data, sources, and status live in [`anomaly.yaml`](anomaly.yaml) (loaded below, never
re-typed as literals in this notebook). Decisions made at each step, including the choices
behind the step-6 fit, are logged in [`decisions.md`](decisions.md).

This notebook ships already solved so that `pytest --nbmake` can confirm it runs top to bottom
in CI. To work through it as an exercise yourself, make your own copy first and avoid reading
`checkpoints_def.py` / `solutions/` until you want a hint or to check your answer. Each step opens
with a **Predict first** question: write your answer down before running the cells.

Claims are tagged *(computed)* when a cell in this notebook produces them, and *(cited)* when they
come from the literature.

In [1]:
import math
import sys
from pathlib import Path

import sympy as sp
from IPython.display import Markdown, Math, display

# nbclient/nbmake set the kernel's cwd to this notebook's own directory; make sure it (and the
# repo root two levels up, for the `anomalies`/`checkpoints`/`fit`/`feynlag_anomalies` packages)
# are importable regardless of how the kernel was launched.
NB_DIR = Path.cwd()
REPO_ROOT = NB_DIR.parents[1]
for p in (NB_DIR, REPO_ROOT):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from feynlag_anomalies.registry import load as load_anomaly
from anomalies.neutrino_mass import checkpoints_def as cd

GEV_TO_EV = 1e9

anomaly = load_anomaly("neutrino_mass")
obs = {o.name: o for o in anomaly.observables}
print(anomaly.title, "--", anomaly.maturity)

Neutrino mass and oscillations -- A3


## What is measured

A neutrino produced with flavour $\alpha$ is a superposition of mass eigenstates. If those have
different masses, they pick up different phases while travelling, and the flavour content changes
with distance. In the two-flavour approximation *(cited: the standard oscillation formula, e.g.
the PDG review "Neutrino masses, mixing, and oscillations")*:

$$P(\nu_\alpha \to \nu_\beta) = \sin^2 2\theta \; \sin^2\!\left(\frac{\Delta m^2 L}{4E}\right),
\qquad \Delta m^2 = m_2^2 - m_1^2 ,$$

in natural units, for baseline $L$ and energy $E$. Two lessons follow:

- **Oscillation needs $\Delta m^2 \neq 0$.** Seeing flavour change means at least one neutrino is
  massive. Oscillations measure only mass *differences*, never the absolute scale.
- **The amplitude measures a mixing angle.** The mismatch between flavour and mass states is the
  PMNS matrix, with three angles $\theta_{12}, \theta_{13}, \theta_{23}$ and a phase
  $\delta_{CP}$.

The global fit used throughout this notebook measures two independent splittings, solar
$\Delta m^2_{21}$ and atmospheric $\Delta m^2_{31}$, both far from zero:

In [2]:
rows = ["| observable | value | 1σ | units |", "|---|---|---|---|"]
rows += [f"| {o.name} | {o.value:.4g} | {o.stat_uncertainty:.2g} | {o.units} |" for o in anomaly.observables]
source = anomaly.sources[0]
display(Markdown("\n".join(rows) + f"\n\nSource: {source.identifier}, consulted {source.consulted_on}."))

| observable | value | 1σ | units |
|---|---|---|---|
| Delta m^2_21 (solar) | 7.49e-05 | 1.9e-06 | eV^2 |
| Delta m^2_31 (atmospheric, normal ordering) | 0.002513 | 2e-05 | eV^2 |
| theta_12 | 33.68 | 0.71 | degrees |
| theta_13 | 8.56 | 0.11 | degrees |
| theta_23 | 43.3 | 0.9 | degrees |
| delta_CP | 212 | 34 | degrees |

Source: arXiv:2410.05380v2 (NuFIT 6.0; JHEP 12 (2024) 216; INSPIRE 2838825), Table 1, IC24 with SK atmospheric data, Normal Ordering, consulted 2026-09-25.

The Standard Model has no neutrino mass term, so it predicts both splittings to be exactly zero.
The rest of this notebook asks *why* it has none, and what is the least we must add.

## Step 0 -- back-of-envelope estimate

**Predict first:** neutrino masses are tiny compared with every other fermion's. Suppose the mass
comes from a Yukawa coupling $y$ to the Higgs, with electroweak scale $v$, and is suppressed by
some heavy scale $M$. Which combination of $y$, $v$ and $M$ has units of mass and gets *smaller*
as $M$ grows?

Evaluate it with an $O(1)$ Yukawa, $v = 246\,\text{GeV}$, and a GUT-ish $M \sim 10^{14}\,\text{GeV}$.

In [3]:
from anomalies.neutrino_mass.solutions.step_0 import m_nu_estimate

answer_0 = m_nu_estimate(cd._P0_Y, cd._P0_V, cd._P0_M)
assert cd.check_0(answer_0)

[step_0] correct -- got 6.0516e-10


The answer is

$$m_\nu \sim \frac{y^2 v^2}{M} ,$$

two powers of $v$ (one Higgs per neutrino leg) divided by one power of the heavy scale. How does
it compare with the data? Every neutrino mass is at least as large as the scale set by the
splittings, and $\sqrt{\Delta m^2_{31}}$ is the natural target:

In [4]:
dm2_31 = obs["Delta m^2_31 (atmospheric, normal ordering)"]
m_atm_eV = math.sqrt(dm2_31.value)
print(f"step-0 estimate:     m_nu ~ {answer_0 * GEV_TO_EV:.3g} eV")
print(f"sqrt(Delta m^2_31):  {m_atm_eV:.3g} eV   (computed from NuFIT 6.0)")
print(f"ratio:               {answer_0 * GEV_TO_EV / m_atm_eV:.3g}")

step-0 estimate:     m_nu ~ 0.605 eV
sqrt(Delta m^2_31):  0.0501 eV   (computed from NuFIT 6.0)
ratio:               12.1


An $O(1)$ Yukawa with $M \sim 10^{14}$ GeV lands within about an order of magnitude of the
measured scale *(computed)*. That is the first hint that neutrino masses could be the low-energy
trace of physics far above the electroweak scale.

## Step 1 -- try it with the SM

**Predict first:** can you write a renormalizable (dimension $\le 4$), gauge-invariant neutrino
mass term with only the SM lepton fields? Check the hypercharges below before you answer.

The *minimal* SM lepton content is a lepton doublet `Ll`, a charged-lepton singlet `eR`, and the
Higgs doublet `H`, with **no** right-handed neutrino. Their quantum numbers, read from the field
declarations:

In [5]:
from anomalies.neutrino_mass.solutions.step_1 import build_sm_lepton_fields

Ll, eR, H, groups = build_sm_lepton_fields()
SU2L, U1Y = groups
rows = ["| field | SU(2)$_L$ | $Y$ | components |", "|---|---|---|---|"]
for f in (Ll, eR, H):
    rows.append(f"| `{f.name}` | {f.reps.get(SU2L, 1)} | {f.reps.get(U1Y, 0)} | "
                f"{', '.join(f'`{c}`' for c in f.components)} |")
display(Markdown("\n".join(rows)))

| field | SU(2)$_L$ | $Y$ | components |
|---|---|---|---|
| `Ll` | 2 | -1/2 | `nuL`, `eL` |
| `eR` | 1 | -1 | `eR` |
| `H` | 2 | 1/2 | `Gp`, `H0` |

The charged-lepton mass comes from $\bar L H e_R$: the hypercharges add to
$+\tfrac12 + \tfrac12 - 1 = 0$. The neutrino analogue $\bar L \tilde H \nu_R$ would need a
$\nu_R$ with $Y = 0$, and the SM has none. A Majorana term $L L$ has $Y = -1$ and is an SU(2)
triplet, so no single Higgs can make it neutral.

`feynlag.suggest.suggest_yukawa` checks this exhaustively: it enumerates every invariant
contraction and re-verifies each one (gauge, discrete, hermiticity, mass dimension).

In [6]:
from feynlag import suggest_yukawa

terms_dim4 = suggest_yukawa([Ll, eR], [H], list(groups), max_dim=4)
for t in terms_dim4:
    display(Math(rf"\text{{{t.label} (dim {t.dim})}}:\quad {sp.latex(t.expr)}"))

answer_1 = len(terms_dim4)
assert cd.check_1(answer_1)

<IPython.core.display.Math object>

[step_1] correct -- got 1


Only the charged-lepton Yukawa survives: **no dimension $\le 4$ neutrino-mass term exists** for
this field content *(computed)*. Two protections block it; see
[`docs/protections.md`](../../docs/protections.md):

- `field_content`: there is no $\nu_R$ to pair with $\nu_L$.
- `accidental_symmetry`: assign lepton number $L = 1$ to `Ll` and `eR` and $L = 0$ to `H`. Every
  renormalizable SM term then conserves $L$ automatically, without imposing it. A Majorana mass
  $\nu_L \nu_L$ carries $\Delta L = 2$, so it can appear only once that accidental symmetry is
  broken.

## Step 2 -- EFT before a model

**Predict first:** if you allow one more power of mass in the operator (dimension 5), which fields
would you combine to build a neutral, gauge-invariant term containing $\nu_L \nu_L$? What is its
lepton number?

Raise `max_dim` to 5.

In [7]:
terms_dim5 = suggest_yukawa([Ll, eR], [H], list(groups), max_dim=5)
for t in terms_dim5:
    display(Math(rf"\text{{{t.label} (dim {t.dim})}}:\quad {sp.latex(t.expr)}"))

answer_2 = any("Weinberg" in t.label for t in terms_dim5)
assert cd.check_2(answer_2)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

[step_2] correct -- got True


The new invariant is the **Weinberg operator** $(LH)(LH)/\Lambda$ *(cited: S. Weinberg, Phys. Rev.
Lett. 43 (1979) 1566)*: two lepton doublets and two Higgs doublets. It has $\Delta L = 2$, and it
is the *unique* dimension-5 operator built from SM fields *(computed above: exactly one new term)*.

The expression above is written in doublet components ($G^+$, $H^0$, $e_L$, $\nu_L$). To see what
it does at low energy, put the Higgs in its vacuum. Following the feynlag-models conventions,
$H = \big(G^+,\ (v + h + iG^0)/\sqrt2\big)$, so the vacuum is $G^+ \to 0$, $H^0 \to v/\sqrt2$:

In [8]:
Gp, H0 = H.components
v = sp.Symbol("v", positive=True)
weinberg = next(t for t in terms_dim5 if "Weinberg" in t.label)
at_vacuum = sp.expand(weinberg.expr.subs({Gp: 0, H0: v / sp.sqrt(2)}))
display(Math(r"(LH)(LH)\big|_{\langle H \rangle} = " + sp.latex(at_vacuum)))

<IPython.core.display.Math object>

Every term with a charged lepton or a Goldstone disappears. What survives is a **Majorana mass
term for $\nu_L$ alone, proportional to $v^2/2$** *(computed)*, plus its conjugate. So after
electroweak symmetry breaking the Weinberg operator gives

$$m_\nu \sim \frac{C\, v^2}{\Lambda} ,$$

with $C$ a dimensionless Wilson coefficient. **What scale $\Lambda$ does it need?** Take
$m_\nu \simeq \sqrt{\Delta m^2_{31}}$ from the data. The heaviest light neutrino has
$m_3 \ge \sqrt{\Delta m^2_{31}}$, with equality when the lightest one is massless. Set $C = 1$
for now:

In [9]:
from anomalies.neutrino_mass.solutions.step_2 import lambda_estimate

m_nu_atm = m_atm_eV / GEV_TO_EV  # GeV

print(f"step-0 illustrative: m_nu ~ {answer_0:.3g} GeV -> Lambda ~ {lambda_estimate(answer_0, cd._P0_V):.3g} GeV")
Lambda_estimate = lambda_estimate(m_nu_atm, cd._P0_V)
print(f"sqrt(Delta m^2_31):  m_nu ~ {m_nu_atm:.3g} GeV -> Lambda ~ {Lambda_estimate:.3g} GeV   (computed)")

step-0 illustrative: m_nu ~ 6.05e-10 GeV -> Lambda ~ 1e+14 GeV
sqrt(Delta m^2_31):  m_nu ~ 5.01e-11 GeV -> Lambda ~ 1.21e+15 GeV   (computed)


$\Lambda \sim 10^{15}$ GeV *if $C = 1$*. Nothing forces $C$ to be $O(1)$, though: what the data
fix is the ratio $C/\Lambda$. Keep that in mind; step 4 shows what $C$ and $\Lambda$ turn out to be
in an explicit model.

## Step 3 -- from the operator to the fields

**Predict first:** the Weinberg operator is $(LH)(LH)$. At tree level, it must come from
exchanging a heavy particle between two of those fields. Which pairs could the heavy particle
couple to, and what quantum numbers would it need?

There are exactly three tree-level completions *(cited: de Blas, Criado, Pérez-Victoria,
Santiago, arXiv:1711.10391)*, one for each way of pairing the four fields:

| completion | heavy field | spin | SU(2)$_L$ | $Y$ | couples to |
|---|---|---|---|---|---|
| type I | $N$ ($\nu_R$) | ½ | 1 | 0 | $\bar L \tilde H N$ |
| type II | $\Delta$ | 0 | 3 | 1 | $L L \Delta$, $H H \Delta^\dagger$ |
| type III | $\Sigma$ | ½ | 3 | 0 | $\bar L \tilde H \Sigma$ |

The minimality rule is: fewer fields, smaller representations, fewer parameters, and no ad hoc
symmetries. By that rule, the gauge-singlet $\nu_R$ (type I) wins on every count. This repo
never declares that Lagrangian itself; it imports the already-verified model from
`feynlag-models` by `model_id` and shows the new terms:

In [10]:
from feynlag_models.registry import build, metadata

answer_3 = "seesaw_type1"
assert cd.check_3(answer_3)

print("feynlag-models maturity:", metadata(answer_3)["maturity_level"])
bundle = build(answer_3)
display(Math(r"\mathcal{L}_{\rm Yuk} = " + sp.latex(bundle.extra["LYukD"])))
display(Math(r"\mathcal{L}_{\rm Maj} = " + sp.latex(bundle.extra["LMaj"])))

[step_3] correct -- got 'seesaw_type1'
feynlag-models maturity: 2


<IPython.core.display.Math object>

<IPython.core.display.Math object>

The Dirac Yukawa $y_\nu$ ties $\nu_R$ to $\nu_L$ through the Higgs, just like the charged-lepton
Yukawa. The Majorana mass $M_R\, \nu_R^T \mathcal{C} \nu_R$ is allowed **only** because $\nu_R$ is
a complete gauge singlet. This is where lepton number is broken.

## Step 4 -- predict before running

**Predict first:** with exactly one $\nu_R$ (one generation), how many physical Majorana mass
eigenstates should the seesaw mechanism produce? And roughly how heavy is each one, if
$m_D \ll M_R$? (Hint: think about the rank and the eigenvalues of the $2\times2$ matrix
$\begin{pmatrix}0 & m_D \\ m_D & M_R\end{pmatrix}$.)

In [11]:
answer_4 = len(bundle.extra["masses"])
check_4 = cd.make_check_4(bundle)
assert check_4(answer_4)

[step_4] correct -- got 2


In [12]:
display(Math(r"\mathcal{M}_\nu = " + sp.latex(bundle.extra["Mnu"])))
display(Math(r"m_\nu^{\rm light} \simeq " + sp.latex(bundle.extra["m_light_approx"])))

values = bundle.values()
bench = bundle.benchmark
light_mass = values[bundle.extra["MN1"].s]
heavy_mass = values[bundle.extra["MN2"].s]
light_approx = float(bundle.extra["m_light_approx"].subs(values))
print(f"benchmark: y_nu = {bench['yv']:.0e}, M_R = {bench['MR']:.0f} GeV")
print(f"light Majorana mass (exact, Takagi):     {light_mass * GEV_TO_EV:.6g} eV   (computed)")
print(f"light Majorana mass (seesaw formula):    {light_approx * GEV_TO_EV:.6g} eV   (computed)")
print(f"relative difference of |m|:              {abs(abs(light_approx) - light_mass) / light_mass:.1e}")
m_D = bench["yv"] * bench["v"] / math.sqrt(2)
print(f"(m_D / M_R)^2:                           {(m_D / bench['MR']) ** 2:.1e}")
print(f"heavy Majorana mass:                     {heavy_mass:.6g} GeV")

<IPython.core.display.Math object>

<IPython.core.display.Math object>

benchmark: y_nu = 1e-06, M_R = 1000 GeV
light Majorana mass (exact, Takagi):     0.0303118 eV   (computed)
light Majorana mass (seesaw formula):    -0.0303118 eV   (computed)
relative difference of |m|:              3.1e-14
(m_D / M_R)^2:                           3.0e-14
heavy Majorana mass:                     1000 GeV


The exact diagonalisation and the seesaw formula differ by a few parts in $10^{14}$, which is
exactly the size of $(m_D/M_R)^2$, the first correction the seesaw expansion drops *(computed)*. The light mass is $m_D^2/M_R$: the heavier $\nu_R$ is, the
lighter $\nu_L$ becomes, hence *seesaw*.

The **minus sign** in $-v^2 y_\nu^2/(2M_R)$ is not a negative mass. For a Majorana fermion, the
sign is a phase that a field redefinition ($\nu \to i\nu$) removes; the physical mass is $|m|$,
which is what Takagi factorisation returns.

**Closing the loop with step 2.** The light mass $y_\nu^2 v^2/(2M_R)$ has exactly the form the
Weinberg operator produced at the vacuum ($\propto v^2/2$), with $C/\Lambda = y_\nu^2/M_R$.
Integrating out $\nu_R$ *is* the Weinberg operator. So what scale does this benchmark
correspond to?

In [13]:
Lambda_eff = bench["MR"] / bench["yv"] ** 2
print(f"type-I benchmark: M_R / y_nu^2 = {Lambda_eff:.3g} GeV   (computed)")
print(f"step 2, C = 1:    Lambda       = {Lambda_estimate:.3g} GeV   (computed)")

type-I benchmark: M_R / y_nu^2 = 1e+15 GeV   (computed)
step 2, C = 1:    Lambda       = 1.21e+15 GeV   (computed)


Both come out around $10^{15}$ GeV *(computed)*, yet the benchmark's $\nu_R$ weighs only 1 TeV.
The "$\Lambda \sim 10^{15}$ GeV" of step 2 is the **combination** $M_R/y_\nu^2$. It can be a GUT-scale
$\nu_R$ with $y_\nu \sim 1$, or a TeV $\nu_R$ with $y_\nu \sim 10^{-6}$. The neutrino masses alone
cannot tell these apart.

## Step 5 -- break it on purpose

**Predict first:** with three lepton generations and one $\nu_R$, the light-neutrino mass matrix
is $m_\nu = -m_D M_R^{-1} m_D^T$, with $m_D$ a $3\times1$ column. What is the rank of this
$3\times3$ matrix? How many light neutrinos are massive, and how many independent $\Delta m^2$ can
it produce?

Compute it with a generic, fully symbolic $m_D$, using feynlag's own seesaw formula
(`feynlag.seesaw_light_mass`). This is plain linear algebra, not a model:

In [14]:
from anomalies.neutrino_mass.solutions.step_5 import light_rank

answer_5 = light_rank(1)
assert cd.check_5(answer_5)
for n in (1, 2):
    r = light_rank(n)
    print(f"{n} nu_R: rank {r} -> {r} massive, {3 - r} massless light neutrino(s)   (computed)")

[step_5] correct -- got 1


1 nu_R: rank 1 -> 1 massive, 2 massless light neutrino(s)   (computed)


2 nu_R: rank 2 -> 2 massive, 1 massless light neutrino(s)   (computed)


With one $\nu_R$, $m_D M_R^{-1} m_D^T$ is an outer product of a single column, so it has rank 1.
That means one massive light neutrino and two massless ones, and therefore only **one**
$\Delta m^2$ *(computed)*. The data show two. **The single-$\nu_R$ seesaw is ruled out** by
oscillations alone. We need at least two $\nu_R$, which give rank 2 *(computed)*: two massive
light neutrinos and one exactly massless.

That gap was written up as a model request rather than built here: this repo never declares a
Lagrangian, and no change is made to `feynlag-models` without approval. See
[`model_requests/seesaw_type1_nN.md`](../../model_requests/seesaw_type1_nN.md).

## Step 6 -- confront with data: a stage-1 $\chi^2$

The two-$\nu_R$ model the step-5 request asked for now exists in `feynlag-models` as
`seesaw_type1_2n` (three lepton generations, two $\nu_R$, maturity L2). Its light-neutrino
matrix has rank 2, so it predicts one **massless** neutrino and two massive ones.

We fit its six Dirac Yukawas $y^\nu_{a b}$ to five measured observables in normal ordering:
$\Delta m^2_{21}$, $\Delta m^2_{31}$, $\theta_{12}$, $\theta_{13}$ and $\theta_{23}$. The heavy
masses $M_1 = 1$ TeV and $M_2 = 3$ TeV are held at the model benchmark. The values and their
source (NuFIT 6.0) come from the loaded `Anomaly`. $\delta_{CP}$ is recorded but not fitted,
because the model's Yukawas are real, so it cannot produce CP violation.

**Why a good fit is guaranteed.** The Casas–Ibarra parametrisation *(cited: Ibarra & Ross,
arXiv:hep-ph/0312138v2, Eq. (6), the regression reference for this fit)* runs the seesaw
backwards. It writes $m_D$ in terms of the light masses, the PMNS matrix, the heavy masses, and a
$3\times2$ orthogonal matrix $R$:
$m_D \sim U \sqrt{\hat m}\, R\, \sqrt{\hat M}$ (schematically). For real Yukawas and fixed $M_{1,2}$,
the count is: two light masses ($m_2, m_3$), three angles, and **one free angle $z$ in $R$**.
That is six parameters, one more than the five observables. Any data point can be matched, with
one direction left over.

Each trial point substitutes the Yukawas into the model's symbolic $5\times5$ mass matrix (below)
and diagonalises it with feynlag's numeric Takagi factorisation. There is no model rebuild per
point.

**Predict first:** six parameters against five observables leaves $-1$ degrees of freedom.
What does a $\chi^2_{\min} \approx 0$ tell you, and what does it *not* tell you?

In [15]:
from anomalies.neutrino_mass.solutions import step_6

model_id = step_6.MODEL_ID
print("feynlag-models maturity:", metadata(model_id)["maturity_level"])
bundle_2n = build(model_id)
display(Math(r"\mathcal{M}_\nu = " + sp.latex(bundle_2n.extra["Mnu"])))
observed = step_6.observed_from_anomaly(anomaly)

fit = step_6.run_fit(bundle_2n, observed)
rows = ["| observable | observed | fitted | pull |", "|---|---|---|---|"]
for name, (value, sigma) in observed.items():
    rows.append(f"| {name} | {value:.5g} ± {sigma:.2g} | {fit['predicted'][name]:.5g} | {fit['pulls'][name]:+.1e} |")
display(Markdown("\n".join(rows)))
print(f"chi2_min = {fit['chi2']:.2e}, ndof = {fit['ndof']}, converged: {fit['success']}, "
      f"perturbative: {fit['perturbative']}   (computed)")

feynlag-models maturity: 2


<IPython.core.display.Math object>

| observable | observed | fitted | pull |
|---|---|---|---|
| Delta m^2_21 (solar) | 7.49e-05 ± 1.9e-06 | 7.49e-05 | -2.9e-09 |
| Delta m^2_31 (atmospheric, normal ordering) | 0.002513 ± 2e-05 | 0.002513 | -2.1e-10 |
| theta_12 | 33.68 ± 0.71 | 33.68 | +6.4e-09 |
| theta_13 | 8.56 ± 0.11 | 8.56 | -6.8e-09 |
| theta_23 | 43.3 ± 0.9 | 43.3 | -1.6e-07 |

chi2_min = 2.65e-14, ndof = -1, converged: True, perturbative: True   (computed)


A $\chi^2_{\min}$ of essentially zero with negative degrees of freedom means the model **can**
accommodate the data. It is not evidence that the data **prefer** it. The model's genuine
prediction is structural: the lightest neutrino is exactly massless, so
$\sum m_\nu = \sqrt{\Delta m^2_{21}} + \sqrt{\Delta m^2_{31}}$ is fixed by the splittings alone.

For contrast, the SM's massless neutrinos predict both splittings to be zero.

In [16]:
m1, m2, m3 = fit["masses_eV"]
print(f"light masses [eV]: m1 = {m1}, m2 = {m2:.4g}, m3 = {m3:.4g}   (computed)")
print(f"sum m_nu = {fit['sum_m_nu_eV']:.4g} eV   (computed; lightest state massless)")
print(f"chi2 over the two splittings: SM = {fit['chi2_sm_dm2']:.3g}, model = {fit['chi2_model_dm2']:.1e}")

answer_6 = fit["predicted"][step_6.FIT_OBSERVABLES[1]] / fit["predicted"][step_6.FIT_OBSERVABLES[0]]
assert cd.make_check_6(anomaly)(answer_6)

light masses [eV]: m1 = 0.0, m2 = 0.008654, m3 = 0.05013   (computed)
sum m_nu = 0.05878 eV   (computed; lightest state massless)
chi2 over the two splittings: SM = 1.73e+04, model = 8.7e-18
[step_6] correct -- got 33.5514


The checkpoint asks for $\Delta m^2_{31}/\Delta m^2_{21} \approx 33.5$. This ratio is the mass
**hierarchy**: the atmospheric splitting is about 30 times the solar one. A rank-1 light sector
has no second splitting, so it cannot produce any ratio at all. Getting it right is the minimum a
two-$\nu_R$ model must do.

**The Yukawas are not unique.** The spare Casas–Ibarra angle $z$ is a flat direction of the
$\chi^2$. Start the fit from a different point (the benchmark with some signs flipped) and it
lands on a different set of Yukawas with the same spectrum:

In [17]:
fit_alt = step_6.run_fit(bundle_2n, observed, x0=step_6.alt_start(bundle_2n))
rows = ["| | " + " | ".join(f"`{k}`" for k in fit["yv"]) + " | $\\chi^2$ | $m_2$, $m_3$ [eV] |",
        "|---" * (len(fit["yv"]) + 3) + "|"]
for label, f in (("benchmark start", fit), ("sign-flipped start", fit_alt)):
    rows.append(f"| {label} | " + " | ".join(f"{y:.2e}" for y in f["yv"].values())
                + f" | {f['chi2']:.1e} | {f['masses_eV'][1]:.5g}, {f['masses_eV'][2]:.5g} |")
display(Markdown("\n".join(rows) + "\n\n*(computed)*"))

| | `yv_e1` | `yv_e2` | `yv_mu1` | `yv_mu2` | `yv_tau1` | `yv_tau2` | $\chi^2$ | $m_2$, $m_3$ [eV] |
|---|---|---|---|---|---|---|---|---|
| benchmark start | 1.59e-07 | 5.40e-07 | 9.05e-07 | -4.50e-07 | 8.92e-07 | 6.39e-07 | 2.7e-14 | 0.0086545, 0.05013 |
| sign-flipped start | 4.42e-08 | -6.01e-07 | 9.39e-07 | -1.09e-07 | -7.13e-07 | 1.13e-06 | 1.8e-14 | 0.0086545, 0.05013 |

*(computed)*

Neutrino-mass data fix $m_D M_R^{-1} m_D^T$, not $m_D$ itself. Pinning down the Yukawas needs
processes that probe $\nu_R$ directly. How visible is $\nu_R$ here? At tree level, it mixes with
$\nu_L$ with angle $\theta_{\alpha k} \simeq (m_D)_{\alpha k}/M_k = y_{\alpha k} v/(\sqrt2 M_k)$:

In [18]:
bench_2n = bundle_2n.benchmark
theta_max = max(abs(y) * bench_2n["v"] / (math.sqrt(2) * bench_2n[f"MR{name[-1]}"])
                for name, y in fit["yv"].items())
print(f"largest active-sterile mixing |theta| ~ {theta_max:.1e}   (computed)")

largest active-sterile mixing |theta| ~ 1.6e-07   (computed)


The mixing is of order $10^{-7}$ *(computed)*. A 1–3 TeV $\nu_R$ is a gauge singlet, so it is
produced and decays only through this mixing, with rates suppressed by $|\theta|^2 \sim 10^{-14}$.
With couplings this small, the model explains the masses but its $\nu_R$ is essentially
out of direct reach. The natural next rungs are
cosmology's bound on $\sum m_\nu$ and neutrinoless double-beta decay's bound on the effective
Majorana mass. Both need sourced limits in `anomaly.yaml` (`tensions_with_other_data` is still
`TODO_VERIFY`), so they are not done here.

## Summary of the ladder

In [19]:
summary = [
    ("0", "dimensional estimate $y^2v^2/M$", f"{answer_0 * GEV_TO_EV:.2g} eV vs $\\sqrt{{\\Delta m^2_{{31}}}}$ = {m_atm_eV:.2g} eV", "computed"),
    ("1", "no dim ≤ 4 $\\nu$ mass term in the SM", f"{answer_1} invariant (charged lepton only)", "computed"),
    ("2", "Weinberg operator is the unique dim-5 term; gives a $\\nu_L$ Majorana mass", f"$\\Lambda/C$ ≈ {Lambda_estimate:.1e} GeV", "computed"),
    ("3", "type-I seesaw is the minimal tree-level completion", "`seesaw_type1`", "cited (arXiv:1711.10391)"),
    ("4", "seesaw reproduces Weinberg with $C/\\Lambda = y^2/M_R$", f"$M_R/y^2$ = {Lambda_eff:.1e} GeV", "computed"),
    ("5", "one $\\nu_R$ gives rank 1: only one $\\Delta m^2$", f"rank {light_rank(1)} (1 $\\nu_R$), {light_rank(2)} (2 $\\nu_R$)", "computed"),
    ("6", "two $\\nu_R$ accommodate the data; $m_1 = 0$", f"$\\chi^2_{{\\min}}$ = {fit['chi2']:.0e} (ndof = {fit['ndof']}), $\\sum m_\\nu$ = {fit['sum_m_nu_eV']:.3g} eV", "computed"),
]
rows = ["| step | result | key number | provenance |", "|---|---|---|---|"]
rows += [f"| {s} | {r} | {n} | {p} |" for s, r, n, p in summary]
display(Markdown("\n".join(rows) + "\n\nNot done yet: cosmology ($\\sum m_\\nu$) and $0\\nu\\beta\\beta$ ($m_{\\beta\\beta}$), pending sourced limits."))

| step | result | key number | provenance |
|---|---|---|---|
| 0 | dimensional estimate $y^2v^2/M$ | 0.61 eV vs $\sqrt{\Delta m^2_{31}}$ = 0.05 eV | computed |
| 1 | no dim ≤ 4 $\nu$ mass term in the SM | 1 invariant (charged lepton only) | computed |
| 2 | Weinberg operator is the unique dim-5 term; gives a $\nu_L$ Majorana mass | $\Lambda/C$ ≈ 1.2e+15 GeV | computed |
| 3 | type-I seesaw is the minimal tree-level completion | `seesaw_type1` | cited (arXiv:1711.10391) |
| 4 | seesaw reproduces Weinberg with $C/\Lambda = y^2/M_R$ | $M_R/y^2$ = 1.0e+15 GeV | computed |
| 5 | one $\nu_R$ gives rank 1: only one $\Delta m^2$ | rank 1 (1 $\nu_R$), 2 (2 $\nu_R$) | computed |
| 6 | two $\nu_R$ accommodate the data; $m_1 = 0$ | $\chi^2_{\min}$ = 3e-14 (ndof = -1), $\sum m_\nu$ = 0.0588 eV | computed |

Not done yet: cosmology ($\sum m_\nu$) and $0\nu\beta\beta$ ($m_{\beta\beta}$), pending sourced limits.